【Python セミナー 第3回 課題】  
#* の横に、このコードはどのような役割をになっているか簡単に記載しています。  

課題: Lasso回帰で、MEDVを予測してください。  
削減する特徴量の数を変えて、予測精度を比較してください。


In [ ]:
#* ライブラリのインポート

# !pip install numpy pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)


In [ ]:
#* 予測結果を評価し、実測値と予測値の散布図を表示する関数を定義する
def calc_scores(targets, predictions):
    #* RMSE・MAE・R2 を計算して、辞書形式で返す
    mse = mean_squared_error(targets, predictions)
    return {
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(targets, predictions),
        'R2': r2_score(targets, predictions),
    }


def pred_plot(train_targets, train_predictions, test_targets, test_predictions, model_name):
    #* トレーニングデータとテストデータの予測精度を計算する
    train_scores = calc_scores(train_targets, train_predictions)
    test_scores = calc_scores(test_targets, test_predictions)

    print(f'\n{model_name} Training Set Performance:')
    print(f"Samples: {len(train_predictions)} | RMSE: {train_scores['RMSE']:.4f} | MAE: {train_scores['MAE']:.4f} | R2: {train_scores['R2']:.4f}")

    print(f'\n{model_name} Test Set Performance:')
    print(f"Samples: {len(test_predictions)} | RMSE: {test_scores['RMSE']:.4f} | MAE: {test_scores['MAE']:.4f} | R2: {test_scores['R2']:.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    #* トレーニングデータの実測値と予測値を散布図で比較する
    ax1.scatter(train_targets, train_predictions, alpha=0.7, color='red', s=24)
    train_min = min(train_targets.min(), np.min(train_predictions))
    train_max = max(train_targets.max(), np.max(train_predictions))
    ax1.plot([train_min, train_max], [train_min, train_max], 'k--', lw=2)
    ax1.set_xlabel('Observed MEDV')
    ax1.set_ylabel('Predicted MEDV')
    ax1.set_title(f"{model_name} Training Set (n={len(train_predictions)})\nRMSE: {train_scores['RMSE']:.3f} | MAE: {train_scores['MAE']:.3f} | R2: {train_scores['R2']:.3f}")
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal', adjustable='box')

    #* テストデータの実測値と予測値を散布図で比較する
    ax2.scatter(test_targets, test_predictions, alpha=0.7, color='blue', s=24)
    test_min = min(test_targets.min(), np.min(test_predictions))
    test_max = max(test_targets.max(), np.max(test_predictions))
    ax2.plot([test_min, test_max], [test_min, test_max], 'k--', lw=2)
    ax2.set_xlabel('Observed MEDV')
    ax2.set_ylabel('Predicted MEDV')
    ax2.set_title(f"{model_name} Test Set (n={len(test_predictions)})\nRMSE: {test_scores['RMSE']:.3f} | MAE: {test_scores['MAE']:.3f} | R2: {test_scores['R2']:.3f}")
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.show()


In [ ]:
#* Lassoで残った特徴量と削減された特徴量を確認する関数を定義する
def make_lasso_coef_table(model, feature_names, threshold=1e-8):
    #* 係数がほぼ0の特徴量を「削減された特徴量」として数える
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': model.coef_,
    })
    coef_df['AbsCoefficient'] = coef_df['Coefficient'].abs()
    coef_df['Selected'] = coef_df['AbsCoefficient'] > threshold
    coef_df['Status'] = np.where(coef_df['Selected'], '使用', '削減')
    return coef_df.sort_values('AbsCoefficient', ascending=False).reset_index(drop=True)


def plot_lasso_coefficients(coef_df, model_name):
    #* 係数が0ではない特徴量を棒グラフで表示し、どの特徴量が残ったかを確認する
    selected_df = coef_df[coef_df['Selected']].copy()

    if selected_df.empty:
        print('係数が0ではない特徴量はありません。')
        return

    plt.figure(figsize=(9, 5))
    sns.barplot(data=selected_df, x='Coefficient', y='Feature', hue='Coefficient', palette='vlag', legend=False)
    plt.axvline(0, color='black', linewidth=1)
    plt.title(f'Non-zero Coefficients in {model_name}')
    plt.xlabel('Coefficient')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()


In [ ]:
#* CSVファイルを読み込んで、Boston Housing データを表として表示する
# 先頭列は sample_1 などのサンプル名なので、index_col=0 で行名として読み込む
df = pd.read_csv('boston.csv', index_col=0)
df


In [ ]:
#* 目的変数を MEDV に設定し、目的変数 y と説明変数 x に分ける
target = 'MEDV'
y = df[target]
x = df.drop(columns=[target])

print(f'目的変数: {target}')
print(f'説明変数の数: {x.shape[1]}')
print(f'データ数: {x.shape[0]}')

x.describe()


In [ ]:
#* データをトレーニングデータ80%、テストデータ20%に分割する
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    shuffle=True,
    random_state=42,
)

#* トレーニングデータの平均と標準偏差を使って、説明変数を標準化する
auto_X_train = (X_train - X_train.mean()) / X_train.std()
auto_X_test = (X_test - X_train.mean()) / X_train.std()

#* 目的変数も標準化して学習し、予測後に元のスケールへ戻す
auto_y_train = (y_train - y_train.mean()) / y_train.std()

print(f'トレーニングセット: {X_train.shape[0]}')
print(f'テストセット: {X_test.shape[0]}')
print(f'MEDV の平均: {y_train.mean():.3f}')
print(f'MEDV の標準偏差: {y_train.std():.3f}')


In [ ]:
#* Lasso の alpha を変えて、削減される特徴量数と予測精度を比較する
# alpha が大きいほど係数が0になりやすく、削減される特徴量が増える
alpha_list = [0.0001, 0.001, 0.003, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
zero_threshold = 1e-8

comparison_rows = []
models = {}
coef_tables = {}

for alpha in alpha_list:
    model_name = f'Lasso alpha={alpha}'

    #* Lasso 回帰モデルを作成し、標準化したトレーニングデータで学習する
    model = Lasso(alpha=alpha, max_iter=100000, random_state=42)
    model.fit(auto_X_train, auto_y_train)

    #* トレーニングデータとテストデータを予測し、MEDV の元のスケールへ戻す
    pred_y_train = model.predict(auto_X_train) * y_train.std() + y_train.mean()
    pred_y_test = model.predict(auto_X_test) * y_train.std() + y_train.mean()

    #* 予測精度を計算する
    train_scores = calc_scores(y_train, pred_y_train)
    test_scores = calc_scores(y_test, pred_y_test)

    #* 係数が0ではない特徴量を「使用された特徴量」として数える
    coef_df = make_lasso_coef_table(model, x.columns, threshold=zero_threshold)
    selected_features = int(coef_df['Selected'].sum())
    reduced_features = int(x.shape[1] - selected_features)

    comparison_rows.append({
        'alpha': alpha,
        'Selected Features': selected_features,
        'Reduced Features': reduced_features,
        'Train RMSE': train_scores['RMSE'],
        'Test RMSE': test_scores['RMSE'],
        'Train MAE': train_scores['MAE'],
        'Test MAE': test_scores['MAE'],
        'Train R2': train_scores['R2'],
        'Test R2': test_scores['R2'],
    })

    models[alpha] = model
    coef_tables[alpha] = coef_df

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
#* 削減した特徴量の数と予測精度の関係をグラフで確認する
plot_df = comparison_df.sort_values(['Reduced Features', 'alpha'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(plot_df['Reduced Features'], plot_df['Test RMSE'], marker='o', color='blue')
ax1.set_xlabel('Reduced Features')
ax1.set_ylabel('Test RMSE')
ax1.set_title('Reduced Features vs Test RMSE')
ax1.grid(True, alpha=0.3)

ax2.plot(plot_df['Reduced Features'], plot_df['Test R2'], marker='o', color='green')
ax2.set_xlabel('Reduced Features')
ax2.set_ylabel('Test R2')
ax2.set_title('Reduced Features vs Test R2')
ax2.grid(True, alpha=0.3)

for _, row in plot_df.iterrows():
    ax1.annotate(row['alpha'], (row['Reduced Features'], row['Test RMSE']), textcoords='offset points', xytext=(4, 4), fontsize=8)
    ax2.annotate(row['alpha'], (row['Reduced Features'], row['Test R2']), textcoords='offset points', xytext=(4, 4), fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
#* テストデータの RMSE が最も小さい Lasso モデルを選ぶ
best_row = comparison_df.loc[comparison_df['Test RMSE'].idxmin()]
best_alpha = float(best_row['alpha'])
best_model = models[best_alpha]
best_model_name = f'Lasso alpha={best_alpha}'

print('テストデータの RMSE が最も小さい条件')
print(best_row.to_string())

#* ベストモデルの予測値を作成し、実測値と予測値の散布図で確認する
best_pred_y_train = best_model.predict(auto_X_train) * y_train.std() + y_train.mean()
best_pred_y_test = best_model.predict(auto_X_test) * y_train.std() + y_train.mean()

pred_plot(y_train, best_pred_y_train, y_test, best_pred_y_test, best_model_name)


In [ ]:
#* ベストモデルで使用された特徴量と削減された特徴量を確認する
best_coef_df = coef_tables[best_alpha]
display(best_coef_df)
plot_lasso_coefficients(best_coef_df, best_model_name)


In [ ]:
#* 結果を文章でまとめるときに使える情報を表示する
selected_names = best_coef_df.loc[best_coef_df['Selected'], 'Feature'].tolist()
reduced_names = best_coef_df.loc[~best_coef_df['Selected'], 'Feature'].tolist()

print(f'この条件では alpha={best_alpha} のとき、テストRMSEが最も小さくなりました。')
print(f'使用された特徴量数: {len(selected_names)} / {x.shape[1]}')
print(f'削減された特徴量数: {len(reduced_names)} / {x.shape[1]}')
print(f'使用された特徴量: {selected_names}')
print(f'削減された特徴量: {reduced_names}')


ここに考察を書く例:  
- alpha を大きくすると、係数が0になる特徴量が増えた。  
- 今回の分割条件では、削減特徴量数が少ない条件でテストRMSEが小さかった。  
- 特徴量を減らしすぎると、説明に必要な情報まで失われてテスト精度が下がった。
